# 4장 2강: 교차검증 전략과 데이터 누수 방지

## 실습 목표

이번 실습에서는 **Credit Card Fraud** 데이터를 이용하여 불균형 분류 문제에서 적절한 교차검증 방법을 선택하고, 데이터 누수를 방지하는 방법을 연습합니다.
이번 실습은 거래 간 시간 의존성을 고려하지 않는 분류 연습으로 StratifiedKFold를 사용하며, 실제 미래 거래 예측을 평가할 때는 시간 순서를 보존해 분할한다.

- KFold 교차검증의 기본 원리를 확인합니다.
- 불균형 분류에서 StratifiedKFold가 필요한 이유를 확인합니다.
- KFold / StratifiedKFold / TimeSeriesSplit의 선택 기준을 구분합니다.
- 잘못된 전처리 순서가 데이터 누수를 만들 수 있음을 이해합니다.
- `Pipeline`을 이용해 전처리와 모델 학습을 교차검증 안에서 안전하게 연결합니다.
- SMOTE를 사용할 때 `imbalanced-learn Pipeline`을 이용해 Train Fold에만 적용되도록 구성합니다.

> 이 실습에서는 4장 2강 교안에 나온 교차검증, 데이터 누수, Pipeline 내용만 사용합니다.

## 실습 환경 / 데이터

- Python
- pandas
- scikit-learn
- imbalanced-learn
- 데이터: `creditcard.csv`

### 주요 컬럼

| 컬럼 | 의미 |
|---|---|
| `V1` ~ `V28` | 거래 정보를 변환한 숫자형 특성 |
| `Class` | 정상 거래 0 / 사기 거래 1 |

이번 실습은 불균형 분류 문제이므로 `Class` 비율을 유지하는 교차검증 전략이 중요합니다.

## 실습 준비

1. 데이터를 불러옵니다.
2. `Class`의 개수와 비율을 확인합니다.
3. `V1`~`V28`을 입력 특성 X로 사용합니다.
4. `Class`를 정답 y로 사용합니다.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    KFold,
    StratifiedKFold,
    TimeSeriesSplit,
    cross_val_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

data = pd.read_csv("creditcard.csv")

features = [f"V{i}" for i in range(1, 29)]

X = data[features]
y = data["Class"]

print("데이터 크기:", data.shape)

print("\n[Class 개수]")
display(y.value_counts().to_frame("count"))

print("\n[Class 비율]")
display(y.value_counts(normalize=True).to_frame("ratio"))

데이터 크기: (284807, 31)

[Class 개수]


,count
Class,
0,284315
1,492



[Class 비율]


,ratio
Class,
0,0.998273
1,0.001727


---

# 필수 1. KFold와 StratifiedKFold 비교

## 배경

KFold는 데이터를 여러 Fold로 나누어 반복 평가합니다.

하지만 Credit Card Fraud처럼 클래스 비율이 크게 불균형한 데이터에서는 Fold마다 사기 거래 비율이 달라질 수 있습니다.

StratifiedKFold는 각 Fold에서 원본 데이터와 비슷한 클래스 비율을 유지하도록 분할합니다.

## 문제 1. KFold의 Fold별 클래스 비율을 확인하세요.

### 요구사항

1. `KFold(n_splits=5, shuffle=True, random_state=42)`를 만듭니다.
2. 각 Fold의 Validation 데이터에서 `Class=1` 비율을 계산합니다.
3. Fold별 사기 거래 비율을 출력합니다.

### 확인 포인트

- KFold는 클래스 비율 자체를 유지하도록 설계된 방법이 아님을 이해했는가?
- Fold별 소수 클래스 비율이 조금씩 달라질 수 있음을 확인했는가?

In [2]:
kfold = KFold(n_splits=5,shuffle=True,random_state=42)
kfold_rows = []
for fold,(train_indices,valid_indices) in enumerate(kfold.split(X,y),start=1):
    valid_y = y.iloc[valid_indices]
    kfold_rows.append({'fold':fold,'valid_size':len(valid_y),'fraud_count':int(valid_y.sum()),'fraud_ratio':valid_y.mean()})
kfold_ratios = pd.DataFrame(kfold_rows)
display(kfold_ratios)
print('전체 사기 비율:',y.mean())

,fold,valid_size,fraud_count,fraud_ratio
0,1,56962,98,0.001720
1,2,56962,93,0.001633
2,3,56961,95,0.001668
3,4,56961,101,0.001773
4,5,56961,105,0.001843


전체 사기 비율: 0.001727485630620034


## 문제 2. StratifiedKFold의 Fold별 클래스 비율을 확인하세요.

### 요구사항

1. `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`를 만듭니다.
2. 각 Validation Fold의 `Class=1` 비율을 계산합니다.
3. 원본 전체 데이터의 사기 거래 비율과 비교합니다.

### 확인 포인트

- 각 Fold의 클래스 비율이 원본과 비슷하게 유지되는가?
- 불균형 분류에서 StratifiedKFold가 적합한 이유를 설명할 수 있는가?

In [3]:
stratified = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
stratified_rows = []
for fold,(train_indices,valid_indices) in enumerate(stratified.split(X,y),start=1):
    valid_y = y.iloc[valid_indices]
    stratified_rows.append({'fold':fold,'valid_size':len(valid_y),'fraud_count':int(valid_y.sum()),'fraud_ratio':valid_y.mean()})
stratified_ratios = pd.DataFrame(stratified_rows)
display(stratified_ratios)
print('전체 사기 비율:',y.mean())

,fold,valid_size,fraud_count,fraud_ratio
0,1,56962,99,0.001738
1,2,56962,99,0.001738
2,3,56961,98,0.001720
3,4,56961,98,0.001720
4,5,56961,98,0.001720


전체 사기 비율: 0.001727485630620034


## 문제 3. 두 교차검증 방법으로 F1-score를 비교하세요.

### 요구사항

1. `LogisticRegression(max_iter=1000)` 모델을 사용합니다.
2. KFold로 5-Fold F1-score를 계산합니다.
3. StratifiedKFold로 5-Fold F1-score를 계산합니다.
4. 각 방법의 Fold별 점수, 평균, 표준편차를 출력합니다.

### 확인 포인트

- 한 번의 점수가 아니라 여러 Fold의 평균과 편차를 확인했는가?
- Credit Card Fraud와 같은 불균형 분류에서는 어떤 방법이 더 안전한 기본 선택인지 설명할 수 있는가?

In [4]:
cv_model = LogisticRegression(max_iter=1000,random_state=42)
kfold_scores = cross_val_score(cv_model,X,y,cv=kfold,scoring='f1')
stratified_scores = cross_val_score(cv_model,X,y,cv=stratified,scoring='f1')
display(pd.DataFrame({'KFold':kfold_scores,'StratifiedKFold':stratified_scores},index=range(1,6)))
cv_comparison = pd.DataFrame({'mean':[kfold_scores.mean(),stratified_scores.mean()],
    'std':[kfold_scores.std(),stratified_scores.std()]},index=['KFold','StratifiedKFold'])
display(cv_comparison)

,KFold,StratifiedKFold
1,0.682927,0.690909
2,0.745342,0.716763
3,0.716049,0.707317
4,0.710843,0.763636
5,0.751323,0.761364


,mean,std
KFold,0.721297,0.024855
StratifiedKFold,0.727998,0.029370


### Q1. Credit Card Fraud 데이터에서는 KFold와 StratifiedKFold 중 어떤 방법을 우선 사용하는 것이 적절한가요?

**답변:**  
시간 의존성을 고려하지 않는 이번 연습에서는 **StratifiedKFold**를 우선 사용합니다. 소수 클래스 비율을 비슷하게 유지하여 일부 Fold에 사기가 너무 적게 들어가는 위험을 줄입니다.

### Q2. 다음 상황에서 어떤 교차검증 방법을 선택할지 작성하세요.

1. 일반적인 회귀 문제
2. 불균형 분류 문제
3. 시간 순서가 중요한 시계열 문제

**답변:**  
1. 독립적인 일반 회귀: KFold.
2. 독립적인 불균형 분류: StratifiedKFold.
3. 시간 순서가 중요한 예측: 날짜 정렬 후 TimeSeriesSplit.

같은 고객·그룹의 중복 관측이 있으면 그룹 단위 분할도 고려해야 합니다.

---

# 필수 2. Pipeline으로 데이터 누수 방지

## 배경

전처리를 전체 데이터에 먼저 `fit()`한 뒤 교차검증하면 Validation Fold의 정보가 전처리 기준에 포함될 수 있습니다.

교안에서는 이를 방지하기 위해 **전처리와 모델을 Pipeline으로 묶는 방법**을 사용합니다.

## 문제 1. StandardScaler와 Logistic Regression을 Pipeline으로 연결하세요.

### 요구사항

1. `StandardScaler()`와 `LogisticRegression(class_weight="balanced", max_iter=1000)`을 하나의 `Pipeline`으로 만듭니다.
2. `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`를 사용합니다.
3. `cross_val_score()`로 F1-score를 계산합니다.
4. Fold별 F1-score, 평균, 표준편차를 출력합니다.

### 확인 포인트

- 각 Fold의 Train 데이터에서만 scaler가 fit되는 구조인가?
- Validation Fold에는 Train에서 학습한 기준으로 transform만 적용되는가?
- 불균형 분류이므로 StratifiedKFold를 사용했는가?

In [5]:
scaled_pipeline = Pipeline([('scaler',StandardScaler()),
    ('model',LogisticRegression(class_weight='balanced',max_iter=1000,random_state=42))])
scaled_scores = cross_val_score(scaled_pipeline,X,y,cv=stratified,scoring='f1')
print('Fold F1:',scaled_scores)
print(f'Mean={scaled_scores.mean():.6f}, Std={scaled_scores.std():.6f}')

Fold F1: [0.12818645 0.12       0.11206897 0.11180124 0.11748634]
Mean=0.117909, Std=0.006026


### Q3. Pipeline을 사용하면 스케일링 누수를 방지할 수 있는 이유는 무엇인가요?

**답변:**  
cross_val_score가 Fold마다 Pipeline을 복제하여 Train Fold로만 scaler.fit과 모델 학습을 수행합니다. Validation Fold에는 Train에서 얻은 평균·표준편차로 transform만 하므로 검증 데이터가 전처리 기준에 들어가지 않습니다.

## 문제 2. SMOTE를 교차검증 안에서 안전하게 적용하세요.

### 배경

SMOTE를 전체 데이터에 먼저 적용한 뒤 교차검증하면 Validation Fold와 매우 비슷한 합성 데이터가 Train에 포함될 수 있습니다.

교안에서는 `imbalanced-learn`의 Pipeline 안에 SMOTE를 넣어 각 Fold의 Train 부분에만 적용하도록 합니다.

### 요구사항

1. `SMOTE(random_state=42)`를 사용합니다.
2. `LogisticRegression(max_iter=1000)`과 함께 `ImbPipeline`으로 연결합니다.
3. `StratifiedKFold`를 사용합니다.
4. `cross_val_score()`로 F1-score를 계산합니다.
5. Fold별 F1-score, 평균, 표준편차를 출력합니다.

### 확인 포인트

- SMOTE가 교차검증 밖에서 전체 데이터에 적용되지 않았는가?
- 각 Fold의 Train 부분에만 SMOTE가 적용되는 구조인가?

In [5]:
smote_pipeline = ImbPipeline([('smote',SMOTE(random_state=42)),
    ('model',LogisticRegression(max_iter=1000,random_state=42))])
smote_scores = cross_val_score(smote_pipeline,X,y,cv=stratified,scoring='f1')
print('Fold F1:',smote_scores)
print(f'Mean={smote_scores.mean():.6f}, Std={smote_scores.std():.6f}')
print('SMOTE는 각 Train Fold에만 fit_resample하며 Validation은 원래 분포로 평가합니다.')

Fold F1: [0.12258953 0.11071429 0.10650888 0.10439883 0.10689869]
Mean=0.110222, Std=0.006511
SMOTE는 각 Train Fold에만 fit_resample하며 Validation은 원래 분포로 평가합니다.


### Q4. SMOTE를 전체 데이터에 먼저 적용한 뒤 교차검증하면 왜 문제가 될 수 있나요?

**답변:**  
합성 샘플 생성에 Validation의 정보가 사용되고 원본과 매우 유사한 합성점이 Train에 들어갈 수 있습니다. 또한 Validation 자체가 인위적으로 균형화되어 실제 분포에서의 성능을 반영하지 못합니다. SMOTE는 ImbPipeline 내부의 Train Fold에서만 적용합니다.

---

# 과제 1. 데이터 특성에 맞는 CV와 누수 없는 Pipeline 구성

## 배경

필수 실습에서는 불균형 분류에 적합한 StratifiedKFold와 Pipeline을 직접 사용했습니다.

이번 과제에서는 같은 Credit Card Fraud 데이터에서 **누수 없는 평가 흐름을 스스로 구성하고 설명**합니다.

> 과제는 필수 실습에서 사용한 구성 요소를 다시 연결하는 수준입니다.

## 요구사항

1. `StandardScaler()`를 사용합니다.
2. `LogisticRegression(class_weight="balanced", max_iter=1000)`을 사용합니다.
3. 두 단계를 `Pipeline`으로 연결합니다.
4. Credit Card Fraud 데이터 특성에 맞는 교차검증 방법을 직접 선택합니다.
5. `n_splits=5`를 사용합니다.
6. `cross_val_score(..., scoring="f1")`로 성능을 평가합니다.
7. Fold별 F1-score, 평균, 표준편차를 출력합니다.
8. 왜 해당 교차검증 방법을 선택했는지 설명합니다.
9. 다음 잘못된 순서를 데이터 누수 관점에서 설명합니다.

```text
전체 X로 StandardScaler.fit_transform()
        ↓
교차검증 수행
```

10. 시계열 데이터라면 같은 방법을 그대로 사용하면 안 되는 이유와 어떤 CV를 선택해야 하는지 작성합니다.

11. `build_pipeline(model)` 함수를 작성하여 `StandardScaler`와 전달받은 모델을 연결한 Pipeline을 반환합니다. 이 함수 안에서는 데이터를 학습하거나 변환하지 않습니다.
12. `evaluate_pipeline(X, y, model, cv, scoring="f1")` 함수를 작성합니다. 11번 함수를 이용해 Pipeline을 만들고 교차검증한 뒤, Fold별 점수·평균·표준편차를 반환합니다. 모델·CV·평가지표를 함수 인자로 교체할 수 있어야 합니다.
13. 12번 함수를 실제 호출하여 요구사항 7번 결과를 출력합니다. CV에서 데이터를 섞는 경우 `shuffle=True, random_state=42`를 지정하고, 모델 등 무작위성이 있는 구성 요소도 가능한 경우 `random_state=42`로 고정합니다.
14. 함수의 재사용 방법과 재현성 설정을 설명하고 Q5~Q8에 답합니다. 별도 `.py` 모듈 제작이나 다른 모델의 추가 학습은 요구하지 않습니다.

### 확인 포인트

- 불균형 분류 특성에 맞는 CV를 선택했는가?
- Pipeline이 전처리 누수를 방지하는 구조임을 설명했는가?
- 평균뿐 아니라 Fold별 점수와 표준편차도 확인했는가?
- TimeSeriesSplit의 사용 상황을 구분할 수 있는가?
- 함수 인자로 모델·CV·평가지표를 바꿀 수 있는가?
- 함수 호출로 전처리·학습·평가가 연결되고 점수가 반환되는가?
- 재현성을 위한 난수 설정을 명시했는가?


In [6]:
def build_pipeline(model):
    # 구조만 생성하며 여기서 데이터를 fit/transform하지 않습니다.
    return Pipeline([('scaler',StandardScaler()),('model',model)])

def evaluate_pipeline(X,y,model,cv,scoring='f1'):
    pipeline = build_pipeline(model)
    scores = cross_val_score(pipeline,X,y,cv=cv,scoring=scoring)
    return {'fold_scores':scores,'mean':float(scores.mean()),'std':float(scores.std())}

task_cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
task_model = LogisticRegression(class_weight='balanced',max_iter=1000,random_state=42)
task_evaluation = evaluate_pipeline(X,y,task_model,task_cv,scoring='f1')
print('Fold F1:',task_evaluation['fold_scores'])
print(f"Mean={task_evaluation['mean']:.6f}, Std={task_evaluation['std']:.6f}")
print('클래스 비율을 유지하려고 StratifiedKFold를 선택합니다. 전체 데이터의 Scaler fit은 검증 평균·분산의 누수를 만듭니다.')
print('미래 예측에서는 날짜 정렬 후 TimeSeriesSplit을 사용해 과거로 학습하고 미래로 검증합니다.')

Fold F1: [0.12818645 0.12       0.11206897 0.11180124 0.11748634]
Mean=0.117909, Std=0.006026
클래스 비율을 유지하려고 StratifiedKFold를 선택합니다. 전체 데이터의 Scaler fit은 검증 평균·분산의 누수를 만듭니다.
미래 예측에서는 날짜 정렬 후 TimeSeriesSplit을 사용해 과거로 학습하고 미래로 검증합니다.


### Q5. 이번 Credit Card Fraud 데이터에서 선택한 CV와 그 이유를 작성하세요.

**답변:**  
StratifiedKFold를 선택했습니다. 사기 비율이 매우 낮아 Fold별 클래스 비율을 유지해야 합니다.

### Q6. 전체 데이터에 StandardScaler를 먼저 fit하면 어떤 정보가 Validation Fold에서 Train 쪽으로 새어 들어갈 수 있나요?

**답변:**  
Validation 특성의 평균과 분산이 Train의 표준화 기준에 들어갑니다. y를 직접 사용하지 않아도 검증 분포 정보를 이용하는 누수입니다. Fold별 Train에만 fit해야 합니다.

### Q7. 작성한 함수를 다른 모델·CV·평가지표에 재사용하려면 어떤 인자를 바꾸면 되나요?

**답변:**
evaluate_pipeline의 model에는 다른 scikit-learn 모델, cv에는 다른 분할기, scoring에는 다른 지표 이름(예: average_precision)을 전달하면 됩니다. build_pipeline은 전달받은 모델을 StandardScaler와 연결하며, scoring과 모델의 예측 방식이 호환되는지 확인해야 합니다.

### Q8. 재현성을 위해 어떤 설정을 고정했으며, Pipeline 생성 함수 안에서 전체 데이터로 전처리를 미리 학습하면 안 되는 이유는 무엇인가요?

**답변:**
CV의 n_splits=5, shuffle=True, random_state=42와 모델·SMOTE의 random_state=42를 고정했습니다. 생성 함수가 전체 X를 fit하면 모든 Validation Fold의 분포를 미리 학습하므로 누수가 생깁니다. 함수는 구조만 반환하고 실제 학습은 CV의 각 Train Fold에 맡깁니다. 같은 데이터·패키지 버전도 재현성에 영향을 줍니다.

---

# 실습 마무리

아래 질문에 짧게 답해보세요.

1. 교차검증을 사용하는 이유는 무엇인가요?  
-> 한 번의 분할에 의존하지 않고 일반화 성능의 평균과 변동을 확인합니다.

2. KFold와 StratifiedKFold의 가장 큰 차이는 무엇인가요?  
-> StratifiedKFold는 Fold별 클래스 비율을 유지합니다.

3. 시계열 데이터에는 어떤 교차검증 방법이 적절한가요?  
-> 시간 정렬 후 TimeSeriesSplit입니다.

4. StandardScaler는 전체 데이터가 아니라 어디에서 fit되어야 하나요?  
-> 각 Train Fold에서만 fit합니다.

5. SMOTE는 각 Fold의 Train / Validation 중 어디에만 적용되어야 하나요?  
->  각 Train Fold에만 적용합니다.

6. Pipeline이 데이터 누수 방지에 도움이 되는 이유는 무엇인가요?  
-> 전처리 학습을 Fold의 Train 안에서 수행하고 Validation에는 변환만 적용합니다.